# Concours MTH3302

## Prédiction de la consommation en carburant de voitures récentes.

### Contexte
Une gestion efficace de la consommation de carburant devient un enjeu crucial tant pour les conducteurs que pour l’industrie automobile, particulièrement dans le contexte actuel de transition énergétique et de réduction des émissions de gaz à effet de serre. La consommation en carburant des véhicules récents dépend de plusieurs caractéristiques techniques telles que la boîte de vitesses, la cylindrée, le nombre de cylindres et le type de transmission. Ces variables influencent directement l'efficacité énergétique et peuvent varier d’un véhicule à l’autre.

### Objectif

Dans cette étude, nous nous concentrons sur la prédiction de la consommation en carburant de voitures récentes. À partir d’un jeu de données comprenant la consommation moyenne en litres pour 100 kilomètres (L/100km) de près de 400 véhicules, ainsi que leurs caractéristiques techniques, l’objectif est de prédire la consommation en carburant pour un ensemble de test en fonction de ces différentes caractéristiques. Ce modèle prédictif permettra d’évaluer plus précisément les performances de consommation des véhicules et d'aider à identifier les facteurs déterminants pour l’optimisation de la consommation de carburant.

### Variables

La variable d'intérêt est la **consommation** en L/100km.

Les variables explicatives sont les suivantes:
- année: l'année du modèle
- type: le type de véhicule
- nombre_cylindres: le nombre de cylindres du moteur
- cylindree: la cylindrée du moteur en L
- transmission: le type de transmission (propulsion, traction, 4x4 et intégrale)
- boite: le type de boite de vitesses (automatique ou manuelle)



In [ ]:
using CSV
using GLM
using MLJ
using DataFrames
using Gadfly
using Random
using Statistics
using Combinatorics
using LinearAlgebra
using HypothesisTests
using StatsModels
using CategoricalArrays
using StatsBase

In [ ]:
train = CSV.read("train.csv", DataFrame, decimal=',')

first(train, 5)

In [ ]:
MLJ.schema(train)

In [ ]:
describe(train)

In [ ]:
# analyser la consommation par cylindree
average_per_cylindree = combine(groupby(train, :cylindree), :consommation => mean)
plot(
    layer(train, x=:cylindree, y=:consommation, Geom.point),
    layer(average_per_cylindree, x=:cylindree, y=:consommation_mean, Geom.line),
    Guide.xlabel("cylindree"), Guide.ylabel("consommation")
)

In [ ]:
# analyser la consommation par cylindree
pp = [plot(train, x=:nombre_cylindres, y=:cylindree, Geom.point),
      plot(train, x=:cylindree, y=:nombre_cylindres, Geom.point)]
p = reshape(pp, (1,2))
set_default_plot_size(30cm, 10cm)
gridstack(p)

In [ ]:
# Afficher des diagrammes en boîte de consommation pour les variables du jeu de données
function box_plot_categorical(data)
    Gadfly.set_default_plot_size(30cm, 35cm)
    p1 = plot(data, x=:annee, y=:consommation, Geom.boxplot, Guide.title("Consommation par Année"), Guide.xlabel("Année"), Guide.ylabel("Consommation"))
    p2 = plot(data, x=:type, y=:consommation, Geom.boxplot, Guide.title("Consommation par Type de voiture"), Guide.xlabel("Type de voiture"), Guide.ylabel("Consommation"))
    p3 = plot(data, x=:nombre_cylindres, y=:consommation, Geom.boxplot, Guide.title("Consommation par nombre de cylindre"), Guide.xlabel("Nombre de cylindre"), Guide.ylabel("Consommation"))
    p4 = plot(data, x=:transmission, y=:consommation, Geom.boxplot, Guide.title("Consommation par type de transmission"), Guide.xlabel("Type de transmission"), Guide.ylabel("Consommation"))
    p5 = plot(data, x=:cylindree, y=:consommation, Geom.point, Geom.smooth(method=:lm), Guide.title("Nuage de points avec la droite de régression"), Guide.xlabel("Cylindree"), Guide.ylabel("Consommation"))
    p6 = plot(data, x=:boite, y=:consommation, Geom.boxplot, Guide.title("Consommation par type de boite de vitesse"), Guide.xlabel("Boite de vitesse"), Guide.ylabel("Consommation"))

    grid = vstack(hstack(p1, p2), hstack(p3, p4), hstack(p5, p6))
    display(grid)

    # réinitialiser la taille pour ne pas affecter les autres graphiques
    Gadfly.set_default_plot_size(20cm, 15cm)
end

In [ ]:
box_plot_categorical(train_data)

In [ ]:
train = CSV.read("train.csv", DataFrame, decimal=',')
test = CSV.read("test.csv", DataFrame, decimal=',');

# Partie 1
## Régressions linéaires simples

In [ ]:
y = train.consommation
n = length(y)

In [ ]:
function compute_residuals(model, y)
    ŷ = StatsModels.predict(model)
    res = (y - ŷ) / std(ŷ)

    return res
end

function plot_explanatory_variable(model, data, xlabel)
    predictions = StatsModels.predict(model)

    Gadfly.plot(
        x = data.x, 
        y = data.y,
        layer(
            x = data.x,
            y = predictions,
            Geom.line,
            Theme(default_color="red"),
        ),
        Geom.point,
        Guide.xlabel(xlabel), 
        Guide.ylabel("Consommation d'essence (L/100km)", orientation=:vertical),
    )
end

function shapiro_wilk_test(model, data)
    errors = compute_residuals(model, data.y)
    p = pvalue(ShapiroWilkTest(errors))

    if p > 0.05
        println("$p > 0.05 -> On accepte l'hypothèse que les données proviennent d'une distribution normale")
    else 
        println("$p ≤ 0.05 -> On rejette l'hypothèse que les données proviennent d'une distribution normale")
    end
end

function residuals_vs_fitted_values_plot_test(model, data)
    errors = compute_residuals(model, data.y)
    fitted_values = fitted(model)

    Gadfly.plot(
        layer(x=fitted_values, y=errors, Geom.point),
        Guide.xlabel("Valeurs prédites"),
        Guide.ylabel("Résidus"),
    )
end

function residuals_vs_observation_order_plot_test(model, data)
    errors = compute_residuals(model, data.y)

    Gadfly.plot(
        layer(x=1:length(errors), y=errors, Geom.point),
        Guide.xlabel("Index"),
        Guide.ylabel("Résidus"),
    )
end

function get_data_set(seed, features, preprocess::Function = data -> nothing)
    Random.seed!(seed)
    data = CSV.read("train.csv", DataFrame, decimal=',')

    preprocess(data)

    train_id = sample(1:nrow(data), round(Int, .8*nrow(data)), ordered=true, replace=false)
    valid_id = setdiff(1:nrow(data), train_id)

    train = data[train_id,:]
    train = remove_outliers(train, features, :consommation)
    valid = data[valid_id,:]

    return train, valid
end

function linear_model(data, features::Vector{Symbol}, target::Symbol=:y)
    formula = Term(target) ~ sum(Term(feature) for feature in filter(x -> x != target, features))

    model = lm(formula, data)

    return model
end

function align_categorical_features(train, valid, features)
    data = vcat(train, valid)

    categorical_features = filter(x -> eltype(data[:, x]) <: AbstractString || eltype(data[:, x]) <: CategoricalValue, features)
    for feature in categorical_features
        train_values = unique(train[!, feature])
        valid_values = unique(valid[!, feature])

        train_indices_to_remove = findall(x -> !(x in valid_values), train[!, feature])
        train = train[setdiff(1:nrow(train), train_indices_to_remove), :]

        valid_indices_to_remove = findall(x -> !(x in train_values), valid[!, feature])
        valid = valid[setdiff(1:nrow(valid), valid_indices_to_remove), :]
    end

    return train, valid
end

## 1.1 Analyse de la variable _nombre_cylindres_

In [ ]:
include("Jeremie_Utils.jl")

In [ ]:
x = float.(train.nombre_cylindres)
data = DataFrame(y = train.consommation, x = x)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

On observe une linéarité entre le nombre de cylindres du véhicule ainsi que sa consommation d'essence 

In [ ]:
plot_explanatory_variable(model, data, "Nombre de cylindres")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont pas distribués normalement 

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

On peut voir que les erreurs ne sont pas constantes selon le nombre de cylindres. Cependant, cela pourrait être causé par la représentation de catégorie de nombre de cylindres disproportionnée.

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

À l'exception de quelques points aberrants, les résidus semblent bien distribués autour de 0.

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

**Signifiance de la variable explicative**

Le nombre de cylindres a un pouvoir explicatif significatif sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.2 Analyse de la variable _type_

In [ ]:
data = DataFrame(y = train.consommation, x = train.type)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

Puisqu'il sagit d'une variable explicative catégorielle nominale, l'hypothèse de linéarité entre les catégories est automatiquement à rejeter

In [ ]:
plot_explanatory_variable(model, data, "Type")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

On peut voir que les erreurs ne sont pas constantes selon le type de voiture.

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

À l'exception de quelques points aberrants, les résidus semblent bien distribués autour de 0.

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

**Signifiance de la variable explicative**

Le type de la voiture a un pouvoir explicatif modéré sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.3 Analyse de la variable _cylindree_

In [ ]:
data = DataFrame(y = train.consommation, x = train.cylindree)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

On voit qu'il existe une relation linéaire entre la cylindrée du moteur et la consommation d'essence de la voiture

In [ ]:
plot_explanatory_variable(model, data, "Cylindrée")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

La variance des erreurs semble relativement constante, mise à part quelques points aberrants et la sur représentation des moteurs avec un cylindrée de 2 litres. 

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

À l'exception de quelques points aberrants, les résidus semblent bien distribués autour de 0.

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

**Signifiance de la variable explicative**

La cylindrée du moteur a un pouvoir explicatif significatif sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.4 Analyse de la variable _transmission_

In [ ]:
x = train.transmission
data = DataFrame(y = train.consommation, x = x)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

Puisqu'il sagit d'une variable explicative catégorielle nominale, et non ordinale, l'hypothèse de linéarité entre les catégories est automatiquement à rejeter

In [ ]:
plot_explanatory_variable(model, data, "Transmission")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

La variance des erreurs semble relativement constante pour chaques transmissions observées

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

Les résidus semblent bien distribués autour de 0.

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

**Signifiance de la variable explicative**

La transmission de la voiture a un pouvoir explicatif modéré sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.5 Analyse de la variable _boite_

In [ ]:
data = DataFrame(y = train.consommation, x = train.boite)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

Puisqu'il sagit d'une variable explicative catégorielle nominale, l'hypothèse de linéarité entre les catégories est automatiquement à rejeter

In [ ]:
plot_explanatory_variable(model, data, "Cylindrée")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

La variance des erreurs semble être plus grande pour les véhicules automatiques que les véhicules manuels

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

Les résidus semblent bien distribués autour de 0.

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

**Signifiance de la variable explicative**

La boîte de la voiture a un pouvoir explicatif très faible sur la consommation d'essence d'une voiture.

In [ ]:
r2(model)

## 1.6 Analyse de la variable _annee_

In [ ]:
x = train.annee
data = DataFrame(y = train.consommation, x = x)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

Il semble avoir une relation linéaire décroissante entre l'année et la consommation d'essence d'un véhicule.

In [ ]:
plot_explanatory_variable(model, data, "Année")

**Vérification de l'hypothèse de normalité des erreurs**

Selon le test de Shapiro-Wilk, la P-valeur est inférieure à 0.05, les résidus ne sont donc pas distribués normalement.

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

La variance des erreurs varie d'année en année

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

Les résidus semblent bien distribués autour de 0.

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

**Signifiance de la variable explicative**

L'année de la voiture a un pouvoir explicatif très faible sur la consommation d'essence.

In [ ]:
r2(model)

# Partie 2
## Régressions linéaires multiples

**Mise en place du seed pour la séparation des données d'entraînement et de validation**

In [ ]:
seed = 9235

### Régression linéaire multiple utilisant toutes les variables explicatives

In [ ]:
features = [:type, :transmission, :nombre_cylindres, :cylindree, :boite, :annee]
train, valid = get_data_set(seed, features)
ols_model = linear_model(train, features, :consommation)

ŷ = float.(StatsModels.predict(ols_model, valid))
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

### Régression linéaire multiple en utilisant que les variables explicatives ayant un pouvoir explicatif respectable

In [ ]:
features = [:type, :transmission, :cylindree]
train, valid = get_data_set(seed, features)
smaller_ols_model = linear_model(train, features, :consommation)

ŷ = float.(StatsModels.predict(smaller_ols_model, valid))
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

### Calcul du VIF pour le modèle de régression linéaire multiple

On obtient un VIF autours de 8-9 pour la cylindrée et le nombre de cylindres, ce qui peut indiquer la présence d'une multicolinéarité problématique. Cela pourrait entraîner un surajustement aux données d'entraînement et augmenter la variance des estimateurs de coefficients de régression. La prédiction est donc plus instable et difficile.

Puisque nous n'avons que très peu de variables explicatives, nous souhaiterions éviter de passer par une analyse des composantes principales, car cela réduirait encore plus notre jeu de données. Nous essaierons plutôt de contrôler cette potentielle multicolinéarité grâce a des techniques de régularisation.

Nous nous attaquerons à ce problème dans la **Partie 3**.

In [ ]:
features = [:type, :transmission, :boite, :nombre_cylindres, :cylindree, :annee]
nominal_features = [:type, :transmission, :boite]
continuous_features = [:nombre_cylindres, :cylindree, :annee]
ordinal_features = []

train, valid = get_data_set(seed, features)
data = vcat(train, valid)
data = encode(data, nominal_features, continuous_features, ordinal_features)
features = map(x -> Symbol(x), get_updated_features(data, features))

p = length(features)
vif = Dict()
for i in 1:p
    model = linear_model(data, continuous_features, features[i])
    vif[features[i]] = 1 / (1 - r2(model))
end

filter((kv) -> kv[2] > 8, vif)

# Partie 3
## Régression linéaire multiple avec technique de régularisation

### Régression ridge avec validation croisée pour choix du lambda optimal

In [ ]:
features = [:type, :transmission, :boite, :nombre_cylindres, :cylindree, :annee]
nominal_features = [:type, :transmission, :boite]
continuous_features = [:nombre_cylindres, :cylindree, :annee]
ordinal_features = []

train, valid = get_data_set(seed, features)
train, valid = align_categorical_features(train, valid, features)

y_train = train[!, :consommation]
X_train = train[!, features]
X_train = encode(X_train, nominal_features, continuous_features, ordinal_features)

X_valid = valid[!, features]
X_valid = encode(X_valid, nominal_features, continuous_features, ordinal_features)

ridge_machine, λ̂ = ridge_regression_cv(X_train, y_train)

ŷ = MLJ.predict(ridge_machine, X_valid)
y_valid = valid[:, :consommation]

println("RMSE: $(rms(ŷ, y_valid))\nλ optimal: $λ̂")

On obtient, en apparence, un résultat de RMSE très comparable a celui de la régression linéaire multiple avec toutes les paramètres. On peut aussi observer que le coefficient le plus important du modèle de régression linéaire multiple, à savoir, la cylindrée, a été pénalisé dans la régression ridge. Cependant, le lambda optimal obtenu est bien petit, donc la régression ridge se comporte davantage comme une régression linéaire.

Tout cela suggère que la multicolinéarité dans les données n'est pas suffisamment élevée pour justifier l'utilisation d'une régularisation importante.

Après avoir écarté l'hypothèse de multicolinéarité, nous nous tournerons vers une approche de régression bayésienne pour tenter d'améliorer les prédiction du modèle de régression linéaire multiple dans la **Partie 4**.

In [ ]:
fitted_params(ridge_machine), ols_model

# Partie 4
## Régression bayésienne

Puisque nous encoderons nos variables explicatives nominales avec l'encodage one hot, il sera difficile de trouver une loi à priori informative pour celle-ci. 

Cependant, comme l'a montré notre analyse préliminaire, la cylindrée et le nombre de cylindres sont deux variables faciles à poser sur une échelle continue et qui ont un bon pouvoir explicatif sur la consommation d'essence. Nous nous concentrerons donc à trouver des lois à priori pour ces deux variables.

### Loi a priori pour la cylindrée

In [ ]:
# source: https://www.kaggle.com/datasets/aishwaryamuthukumar/cars-dataset-audi-bmw-ford-hyundai-skoda-vw
data = CSV.read("cars_dataset_1.csv", DataFrame, decimal=',')
data = filter(x -> x.year >= 2014, data)
data.engine_size = String.(data[!, :engineSize])
data.engine_size = parse.(Float64, data[!, :engine_size])
data = filter(x -> x.engine_size > 1, data)
data.engine_size = log.(data.engine_size)

dist_cylindree = fit(Gamma, data.engine_size)
Gadfly.plot(
    layer(x=data.engine_size, Geom.histogram(density=true)),
    layer(x->pdf(dist_cylindree, x), 0, 2, Theme(default_color=colorant"red")),
)

### Loi a priori pour le nombre de cylindres

Nous utiliserons le logarithme du nombre de cylindre pour réduire l'hétéroscédasticité observé dans l'analyse préliminaire. Cela nous permettra aussi de réduire l'échelle du nombre de cylindre, permettant de donner plus de considération aux valeurs extrêmes

In [ ]:
# source: https://www.kaggle.com/datasets/CooperUnion/cardataset/data
data = CSV.read("cars_dataset_2.csv", DataFrame, decimal=',')
data.engine_cylinders = data[!, "Engine Cylinders"]
data = filter(x -> x.Year >= 2014, data)
data = filter(x -> !ismissing(x.engine_cylinders) && x.engine_cylinders > 1, data)
data.engine_cylinders = map(x -> float(x), data.engine_cylinders)
data.engine_cylinders = log.(data.engine_cylinders)

dist_nombre_cylindres = fit(Gamma, data.engine_cylinders)
Gadfly.plot(
    layer(x=data.engine_cylinders, Geom.histogram(bincount=30, density=true)),
    layer(x->pdf(dist_nombre_cylindres, x), 1, 3, Theme(default_color=colorant"red")),
)